In [9]:
import tensorflow as tf
print("Versão do TF:", tf.__version__)
from tensorflow.keras import layers, models
import numpy as np
import matplotlib.pyplot as plt
import cv2
from google.colab import files

# 1. Carregar Dataset (Fashion MNIST - Roupas)
# Classes: 0:Camiseta, 1:Calça, 2:Pullover, 3:Vestido, 4:Casaco,
#          5:Sandália, 6:Camisa, 7:Tênis, 8:Bolsa, 9:Bota
print("Carregando dataset...")
(train_images, train_labels), (test_images, test_labels) = tf.keras.datasets.fashion_mnist.load_data()

# Normalizar (0-255 -> 0-1) e redimensionar para incluir canal de cor (28, 28, 1)
train_images = train_images.reshape((60000, 28, 28, 1)).astype('float32') / 255
test_images = test_images.reshape((10000, 28, 28, 1)).astype('float32') / 255

# 2. Definir a CNN
model = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(10, activation='softmax')
])

# 3. Treinar
print("Treinando modelo...")
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

model.fit(train_images, train_labels, epochs=5, validation_data=(test_images, test_labels))

print("\n--- Verificando Desempenho do Modelo Original ---")
# 1. Salvar o modelo no formato Keras
model.save('modelo_original.keras')

# 2. Avaliar o desempenho com os dados de teste
loss, accuracy = model.evaluate(test_images, test_labels, verbose=0)
print(f"Acurácia do modelo .keras: {accuracy*100:.2f}%")
print(f"Perda (Loss) do modelo: {loss:.4f}\n")

# 3. Baixar o modelo original (Opcional, mas bom para ter na pasta do projeto)
files.download('modelo_original.keras')

# 4. Converter para TFLite com Full Integer Quantization
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

# 4.1 Criar o "Dataset Representativo"
def representative_data_gen():
  for i in range(100): # Pega 100 imagens de treino para calibrar
    # O yield precisa retornar o formato exato da entrada: (1, 28, 28, 1) em float32
    yield [train_images[i:i+1]]

converter.representative_dataset = representative_data_gen

# 4.2 Forçar operações estritamente inteiras
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]

# 4.3 Forçar a entrada e saída para INT8
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

tflite_model = converter.convert()

# Salvar
with open('model.tflite', 'wb') as f:
    f.write(tflite_model)

model.evaluate(test_images, test_labels)

print("Download dos arquivos...")
files.download('model.tflite')

Versão do TF: 2.19.0
Carregando dataset...
Treinando modelo...
Epoch 1/5
1875/1875 [==============================] - 32s 16ms/step - loss: 0.4554 - accuracy: 0.8410 - val_loss: 0.3669 - val_accuracy: 0.8719
Epoch 2/5
1875/1875 [==============================] - 30s 16ms/step - loss: 0.3201 - accuracy: 0.8877 - val_loss: 0.3193 - val_accuracy: 0.8881
Epoch 3/5
1875/1875 [==============================] - 29s 16ms/step - loss: 0.2826 - accuracy: 0.9010 - val_loss: 0.2966 - val_accuracy: 0.8935
Epoch 4/5
1875/1875 [==============================] - 30s 16ms/step - loss: 0.2596 - accuracy: 0.9081 - val_loss: 0.2959 - val_accuracy: 0.8906
Epoch 5/5
1875/1875 [==============================] - 29s 16ms/step - loss: 0.2422 - accuracy: 0.9142 - val_loss: 0.2815 - val_accuracy: 0.8970

--- Verificando Desempenho do Modelo Original ---
Acurácia do modelo .keras: 89.70%
Perda (Loss) do modelo: 0.2815



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/convert.py:854: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


313/313 [==============================] - 2s 6ms/step - loss: 0.2815 - accuracy: 0.8970
Download dos arquivos...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [10]:
# Célula para pegar imagens de teste

import tensorflow as tf
import cv2
import numpy as np
from google.colab import files

# 1. Carregar apenas o dataset de teste
print("Carregando dataset...")
(_, _), (test_images, test_labels) = tf.keras.datasets.fashion_mnist.load_data()

# Nomes das classes para colocar no nome do arquivo
class_names = ['Camiseta', 'Calca', 'Pullover', 'Vestido', 'Casaco',
               'Sandalia', 'Camisa', 'Tenis', 'Bolsa', 'Bota']

# 2. Escolher os índices das imagens que você quer baixar do conjunto de teste
# (Pode mudar esses números para pegar imagens diferentes de 0 a 9999)
indices_para_salvar = [0, 1, 2, 3, 5, 7, 8, 9]

print("Gerando e baixando imagens...")
for idx in indices_para_salvar:
    img = test_images[idx]
    label = test_labels[idx]
    nome_classe = class_names[label]

    nome_arquivo = f'teste_{idx}_{nome_classe}.png'

    # Salva e faz o download
    cv2.imwrite(nome_arquivo, img)
    files.download(nome_arquivo)
    print(f"Baixado: {nome_arquivo}")

Carregando dataset...
Gerando e baixando imagens...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Baixado: teste_0_Bota.png


<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

Baixado: teste_8_Sandalia.png


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Baixado: teste_9_Tenis.png
